# Quality decision

Notebook 03 predicts the properties of a weld; this notebook answers the initial question, *is this weld qualitative?* (plan: `doc/06_quality_decision_plan.md`). A weld is qualitative when it meets a **specification**, a set of thresholds taken from a class of consumable (`src/specification.py`). The decision is taken per property: each prediction gets an **interval calibrated by held-out study** (`src/decision.py`), and a property is *conforming* when the whole interval is above the threshold, *non conforming* when the whole interval is below, *to test* otherwise. The model is the CatBoost of notebook 03 (rounds tuned), predicted by leave-one-study-out, so every weld is predicted by a model that never saw its study.

## 0. Setup

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "src" / "config.py").exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.cleaning import clean_dataframe
from src.config import RESULTS_DIR, TARGETS
from src.decision import (LEVEL, add_intervals, decide_welds, decision_report, interval_coverage,
                          run_predictions, study_quantiles)
from src.loading import load_raw
from src.specification import SPECIFICATIONS

OUT = RESULTS_DIR / "quality_decision"
OUT.mkdir(parents=True, exist_ok=True)
RERUN = False      # True recomputes the leave-one-study-out predictions (about 3 minutes)

clean = clean_dataframe(load_raw())
predictions = run_predictions(clean, OUT, rerun=RERUN, log=print)
print(predictions.shape, "| held-out studies:", predictions["study"].nunique(), "| nominal coverage of the intervals:", LEVEL)

(7539, 6) | held-out studies: 10 | nominal coverage of the intervals: 0.8


## 1. Intervals calibrated by study

For each held-out study, the half-width of the interval is a quantile of the leave-one-study-out residuals of the **other** studies, each study weighing the same (`study_quantiles`). The nominal coverage is 80%, raised by the factor (K+1)/K of split conformal prediction for K calibration studies. With 5 to 7 studies the guarantee is weak: the coverage is read study by study.

In [2]:
quantiles = study_quantiles(predictions, LEVEL)
intervals = add_intervals(predictions, quantiles)
coverage = interval_coverage(intervals)
coverage.to_csv(OUT / "06_interval_coverage.csv", index=False)

by_target = coverage.groupby("target").agg(studies=("study", "size"), half_width=("width", lambda w: w.mean() / 2),
                                           mean_coverage=("coverage", "mean"), worst_study_coverage=("coverage", "min"))
by_target["spread of the data (std)"] = clean[by_target.index].std().round(1)
print("interval half-width (physical units) and coverage of the measured welds:")
print(by_target.reindex(TARGETS).round(2).to_string())
print("\ncoverage of each held-out study:")
print(coverage.pivot(index="study", columns="target", values="coverage")[TARGETS].round(2).to_string())

interval half-width (physical units) and coverage of the measured welds:
                       studies  half_width  mean_coverage  worst_study_coverage  spread of the data (std)
target                                                                                                   
yield_strength_mpa           7      108.79           0.92                  0.65                      92.9
uts_mpa                      7       90.39           0.92                  0.61                      88.6
elongation_pct               7        7.83           0.93                  0.80                       4.9
reduction_of_area_pct        7       13.39           0.90                  0.52                       8.9
charpy_toughness_j           5      105.09           1.00                  1.00                      50.1

coverage of each held-out study:
target       yield_strength_mpa  uts_mpa  elongation_pct  reduction_of_area_pct  charpy_toughness_j
study                                              

**Your conclusions:**

- **The intervals are wider than the spread of the data itself.** Half-widths: +/-109 MPa on yield strength (standard deviation of the data 93), +/-90 MPa on UTS (89), +/-7.8 points on elongation (4.9), +/-13.4 points on reduction of area (8.9) and +/-105 J on Charpy toughness (50). That is the honest size of the error on a new study (notebook 03), not a defect of the construction.
- **The coverage is above the nominal 80% on average** (0.90-0.93; Charpy 1.00 because its intervals cover almost any energy), the factor (K+1)/K and the cap at the largest residual making the intervals conservative.
- **It fails on one study, as expected from notebook 03**: `Wolst` is covered at 0.65 on yield strength, 0.61 on UTS and 0.52 on reduction of area, and `Gar&K` at 0.75-0.88 on four targets. With 5 to 7 studies the calibration does not protect against a study that behaves differently from all the others.

## 2. Decision per specification, against the measured conformity

For each specification of the registry, the weld is decided from the intervals of the properties it requires (the Charpy energy predicted at the temperature of the requirement). The calls are compared with the **measured** conformity (`Specification.conformity`), on the welds whose conformity is known (all required properties reported, the Charpy at an informative temperature) and whose study was held out. The interval rule is compared with the **point rule** of a user who ignores the uncertainty (the prediction itself against the threshold). The costly error is the **false accept**: a weld called conforming that does not conform.

In [3]:
rows = []
for name, spec in SPECIFICATIONS.items():
    labels = spec.conformity(clean)
    for rule, point in (("interval", False), ("point", True)):
        report = decision_report(decide_welds(clean, intervals, spec, point=point), labels, clean["source"]).iloc[0]
        rows.append({"specification": name, "rule": rule, **report.drop("study").to_dict()})
summary = pd.DataFrame(rows)
summary["share conforming (base rate)"] = (summary["true_conforming"] / summary["welds"]).round(2)
summary["precision of 'conforming'"] = (1 - summary["false_accepts"] / summary["called_conforming"]).round(2)
summary["precision of 'non conforming'"] = (1 - summary["false_rejects"] / summary["called_non_conforming"]).round(2)
summary.to_csv(OUT / "06_decision_summary.csv", index=False)
print(summary.to_string(index=False))

   specification     rule  welds  true_conforming  called_conforming  false_accepts  to_test  called_non_conforming  false_rejects  share conforming (base rate)  precision of 'conforming'  precision of 'non conforming'
ISO 2560-A E42 4 interval    335               34                  0              0      335                      0              0                          0.10                        NaN                            NaN
ISO 2560-A E42 4    point    335               34                115             81       85                    135              0                          0.10                       0.30                           1.00
  AWS A5.1 E7018 interval    548              392                  0              0      548                      0              0                          0.72                        NaN                            NaN
  AWS A5.1 E7018    point    548              392                523            133        0                     25         

**Your conclusions:**

- **The model cannot certify the stricter classes, with or without intervals.** With the interval rule, `E7018`, `E7018-1` and `ISO 2560-A E42 4` accept **no** weld: all 548, 214 and 335 are *to test*. The thresholds (400 MPa, 22%) are closer to the typical values (mean yield strength 509, elongation 26) than the half-widths (+/-109 MPa, +/-7.8), so no interval can clear them.
- **The point rule, which ignores the uncertainty, accepts almost everything and is wrong when it matters**: 133 false accepts out of 523 welds called conforming for `E7018` (precision 0.75 for a base rate of 0.72), 78 out of 107 for `E7018-1` (0.27 against 0.14) and 81 out of 115 for `ISO 2560-A E42 4` (0.30 against 0.10). Its precision is hardly better than accepting at random.
- **On lenient classes the interval rule is safe but gains little.** `E6013` (97% of the decided welds conform): 352 accepted out of 543, 9 false accepts, a precision of 0.97 equal to the base rate, so no better than accepting everything (the point rule: 19 false accepts out of 543). `E6010` (base rate 0.81): the interval rule accepts only 16 welds with one false accept (0.94), the point rule 525 with 82 false accepts (0.84).
- **The reliable use is the other side: flagging welds that do not conform.** When the point rule calls a weld *non conforming* it is right 92% of the time on `E7018` (25 calls, 2 wrong), 91% on `E6010` and 100% on `E7018-1` (23 calls) and `ISO 2560-A E42 4` (135 calls), which find 12%, 15% and 45% of the non-conforming welds respectively (23 of 185, 23 of 156 and 135 of 301). It is a screening that spares tests, not a certification.

### 2.1 Study by study, and the effect of the coverage level

In [4]:
spec = SPECIFICATIONS["AWS A5.1 E6013"]
labels = spec.conformity(clean)
print("AWS A5.1 E6013, interval rule, per held-out study:")
print(decision_report(decide_welds(clean, intervals, spec), labels, clean["source"]).iloc[1:].to_string(index=False))

sensitivity = []
for level in (0.5, 0.8, 0.9):
    levelled = add_intervals(predictions, study_quantiles(predictions, level))
    for name in ("AWS A5.1 E7018", "AWS A5.1 E6013"):
        spec = SPECIFICATIONS[name]
        report = decision_report(decide_welds(clean, levelled, spec), spec.conformity(clean), clean["source"]).iloc[0]
        sensitivity.append({"level": level, "specification": name, **report.drop("study").to_dict()})
sensitivity = pd.DataFrame(sensitivity)
sensitivity.to_csv(OUT / "06_level_sensitivity.csv", index=False)
print("\neffect of the nominal coverage level:")
print(sensitivity.to_string(index=False))

AWS A5.1 E6013, interval rule, per held-out study:
      study  welds  true_conforming  called_conforming  false_accepts  to_test  called_non_conforming  false_rejects
       Cunh     31               31                 19              0       12                      0              0
      Evans    330              321                212              0      118                      0              0
EvansLetter     51               49                 43              2        8                      0              0
      Gar&K     42               34                 26              7       16                      0              0
  KocakPREv     21               21                 21              0        0                      0              0
    RR82011     45               45                 28              0       17                      0              0
      Wolst     23               23                  3              0       20                      0              0

effect of th

**Your conclusions:**

- **The weakest study is where the false accepts are**: for `E6013`, `Gar&K` has 7 false accepts out of 26 welds called conforming and `EvansLetter` 2 out of 43; `Wolst` has only 3 acceptances out of 23 welds, because its intervals are the widest relative to the margin.
- **There is no free lunch in the level of the interval.** At a nominal coverage of 0.5 the rule accepts 317 `E7018` welds with 51 false accepts (precision 0.84 for a base rate of 0.72); at 0.8 it accepts none; at 0.9 it accepts none, `E6013` included (the factor (K+1)/K then caps the interval at the largest residual of the other studies). Accepting more welds means accepting more bad ones, and the cost ratio between the two errors (not known) decides where to stop.
- **What would change the picture is the level error, not the model.** The intervals are wide because a new study is placed at the wrong level (notebook 03: the models order the welds of a study well and place its level badly). A few measured welds of the new study would recentre the predictions; that is the step of `doc/06_quality_decision_plan.md` (point 6) that this notebook does not test.